In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 11
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


In [2]:
# Check file sizes before downloading
QOG_BASE = "https://www.qogdata.pol.gu.se/data"

def get_latest_qog_version():
    """Auto-detect latest QoG version by trying recent years."""
    current_year = datetime.today().year
    for year in range(current_year, current_year - 3, -1):
        yy = str(year)[-2:]
        url = f"{QOG_BASE}/qog_std_ts_jan{yy}.csv"
        response = requests.head(url, timeout=10, allow_redirects=True)
        if response.status_code == 200 and 'text/csv' in response.headers.get('Content-Type', ''):
            size_mb = int(response.headers.get('Content-Length', 0)) / (1024*1024)
            print(f"Latest QoG version: Jan{yy} ({year})")
            return yy, year
    return None, None

QOG_VERSION, QOG_YEAR = get_latest_qog_version()

# Check sizes of both Standard and Basic datasets
for dataset in ['std', 'bas']:
    url = f"{QOG_BASE}/qog_{dataset}_ts_jan{QOG_VERSION}.csv"
    response = requests.head(url, timeout=10, allow_redirects=True)
    size_mb = int(response.headers.get('Content-Length', 0)) / (1024*1024)
    print(f"qog_{dataset}_ts_jan{QOG_VERSION}.csv: {size_mb:.1f} MB")

Latest QoG version: Jan26 (2026)
qog_std_ts_jan26.csv: 68.3 MB
qog_bas_ts_jan26.csv: 14.9 MB


In [4]:
# Download Standard dataset — contains all sources
print("Downloading QoG Standard Time-Series dataset...")
url = f"{QOG_BASE}/qog_std_ts_jan{QOG_VERSION}.csv"
response = requests.get(url, timeout=300)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024/1024:.1f} MB")

qog_std = pd.read_csv(io.StringIO(response.text), low_memory=False)
print(f"\nShape: {qog_std.shape}")
print(f"Years: {qog_std['year'].min()} — {qog_std['year'].max()}")
print(f"Countries: {qog_std['cname'].nunique()}")

# Re-check source prefixes in Standard dataset
print("\nSource prefix check in Standard dataset:")
for prefix in needed_prefixes:
    cols = [c for c in qog_std.columns if c.startswith(prefix)]
    print(f"  {prefix}: {len(cols)} variables — {cols[:3]}")

Status: 200, Size: 68.3 MB

Shape: (12585, 1813)
Years: 1946 — 2025
Countries: 204

Source prefix check in Standard dataset:
  dr_: 4 variables — ['dr_eg', 'dr_ig', 'dr_pg']
  gd_: 3 variables — ['gd_ptsa', 'gd_ptsh', 'gd_ptss']
  pts_: 0 variables — []
  gpi_: 4 variables — ['gpi_conf', 'gpi_gpi', 'gpi_mil']
  p_pol: 1 variables — ['p_polity2']
  ibp_: 4 variables — ['ibp_cat', 'ibp_leg', 'ibp_obi']
  gain_: 18 variables — ['gain_cap', 'gain_econ', 'gain_ecos']
  epi_: 11 variables — ['epi_agr', 'epi_bdh', 'epi_cch']
  bci_: 2 variables — ['bci_bci', 'bci_bcistd']
  lld_: 2 variables — ['lld_capacity', 'lld_capstd']
  romelli_: 0 variables — []
  ccp_: 18 variables — ['ccp_buildsoc', 'ccp_cc', 'ccp_childwrk']
  nelda_: 10 variables — ['nelda_fme', 'nelda_mbbe', 'nelda_mtop']
  pei_: 2 variables — ['pei_eir_1', 'pei_peii_1']
  pg_: 0 variables — []
  gra_: 0 variables — []
  ideaesd_: 6 variables — ['ideaesd_esf', 'ideaesd_esnl', 'ideaesd_esp']


In [5]:
# Check alternative prefixes for missing sources
alt_checks = {
    'romelli': 'romelli',
    'IRENA/pg': 'irena',
    'climate laws/gra': 'ccl',
    'KOF trade specific': 'dr_trad',
    'IBP open budget score': 'ibp_obi',
}

for name, prefix in alt_checks.items():
    cols = [c for c in qog_std.columns if prefix.lower() in c.lower()]
    print(f"  {name} ('{prefix}'): {len(cols)} — {cols[:5]}")

# Also check what's available for dr (KOF) in full
print("\nAll dr_ columns:")
print([c for c in qog_std.columns if c.startswith('dr_')])

# And ibp_ columns in full
print("\nAll ibp_ columns:")
print([c for c in qog_std.columns if c.startswith('ibp_')])

  romelli ('romelli'): 0 — []
  IRENA/pg ('irena'): 0 — []
  climate laws/gra ('ccl'): 0 — []
  KOF trade specific ('dr_trad'): 0 — []
  IBP open budget score ('ibp_obi'): 1 — ['ibp_obi']

All dr_ columns:
['dr_eg', 'dr_ig', 'dr_pg', 'dr_sg']

All ibp_ columns:
['ibp_cat', 'ibp_leg', 'ibp_obi', 'ibp_pub']


In [6]:
# Inspect key multi-variable sources to identify relevant columns
sources_to_inspect = {
    'KOF (dr_)': [c for c in qog_std.columns if c.startswith('dr_')],
    'ND-GAIN (gain_)': [c for c in qog_std.columns if c.startswith('gain_')],
    'EPI (epi_)': [c for c in qog_std.columns if c.startswith('epi_')],
    'CCP (ccp_)': [c for c in qog_std.columns if c.startswith('ccp_')],
    'NELDA (nelda_)': [c for c in qog_std.columns if c.startswith('nelda_')],
    'IDEA (ideaesd_)': [c for c in qog_std.columns if c.startswith('ideaesd_')],
    'IBP (ibp_)': [c for c in qog_std.columns if c.startswith('ibp_')],
    'GPI (gpi_)': [c for c in qog_std.columns if c.startswith('gpi_')],
    'PEI (pei_)': [c for c in qog_std.columns if c.startswith('pei_')],
}

for source, cols in sources_to_inspect.items():
    print(f"\n{source}:")
    for col in cols:
        non_null = qog_std[col].notna().sum()
        latest_yr = qog_std[qog_std[col].notna()]['year'].max()
        print(f"  {col}: {non_null} non-null obs, latest year {latest_yr}")


KOF (dr_):
  dr_eg: 8982 non-null obs, latest year 2023
  dr_ig: 9139 non-null obs, latest year 2023
  dr_pg: 9272 non-null obs, latest year 2023
  dr_sg: 9272 non-null obs, latest year 2023

ND-GAIN (gain_):
  gain_cap: 5099 non-null obs, latest year 2023
  gain_econ: 5324 non-null obs, latest year 2023
  gain_ecos: 5179 non-null obs, latest year 2023
  gain_exp: 5527 non-null obs, latest year 2023
  gain_food: 5440 non-null obs, latest year 2023
  gain_gain: 5382 non-null obs, latest year 2023
  gain_gaingdp: 5324 non-null obs, latest year 2023
  gain_gov: 5418 non-null obs, latest year 2023
  gain_hab: 5527 non-null obs, latest year 2023
  gain_heal: 5469 non-null obs, latest year 2023
  gain_inf: 4867 non-null obs, latest year 2023
  gain_read: 5527 non-null obs, latest year 2023
  gain_readgdp: 5411 non-null obs, latest year 2023
  gain_sens: 5237 non-null obs, latest year 2023
  gain_soc: 5353 non-null obs, latest year 2023
  gain_vuln: 5382 non-null obs, latest year 2023
  gain

In [7]:
QOG_VARIABLES = {
    'cname': 'country_name',
    'ccodecow': 'cow_code',
    'ccodealp': 'country_code',
    'year': 'year',
    'dr_eg': 'kof_economic_globalisation',
    'gd_ptsa': 'pts_amnesty',
    'gd_ptsh': 'pts_hrw',
    'gd_ptss': 'pts_statedept',
    'ibp_obi': 'obs_open_budget_index',
    'gain_gov': 'nd_gain_governance_readiness',
    'gain_read': 'nd_gain_readiness',
    'bci_bci': 'bci_corruption_index',
    'lld_capacity': 'hanson_sigman_state_capacity',
    'ccp_syst': 'ccp_government_system',
    'ccp_market': 'ccp_market_economy_provisions',
    'ccp_civil': 'ccp_civil_rights_provisions',
    'ccp_infoacc': 'ccp_information_access',
    'ccp_equal': 'ccp_equality_provisions',
    'pei_peii_1': 'pei_electoral_integrity_index',
    'gpi_gpi': 'gpi_peace_index',
}

missing_cols = [col for col in QOG_VARIABLES.keys() if col not in qog_std.columns]
if missing_cols:
    print(f"⚠️ Missing columns: {missing_cols}")
else:
    print("All selected columns present ✅")
print(f"\nSelecting {len(QOG_VARIABLES) - 4} indicators plus 4 identifiers")

All selected columns present ✅

Selecting 16 indicators plus 4 identifiers


In [8]:
# Filter to selected columns and rename
qog = qog_std[list(QOG_VARIABLES.keys())].copy()
qog = qog.rename(columns=QOG_VARIABLES)

# Filter to framework start year
qog = qog[qog['year'] >= FRAMEWORK_START_YEAR].copy()

# Sort
qog = qog.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {qog.shape}")
print(f"Years: {qog['year'].min()} — {qog['year'].max()}")
print(f"Countries: {qog['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (qog.isnull().sum() / len(qog) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(qog.head(3))

Shape: (6889, 20)
Years: 1990 — 2025
Countries: 200

Missing values (%):
pei_electoral_integrity_index    92.0
obs_open_budget_index            86.8
pts_hrw                          83.8
gpi_peace_index                  58.2
hanson_sigman_state_capacity     39.1
pts_amnesty                      29.1
nd_gain_governance_readiness     21.4
nd_gain_readiness                19.8
bci_corruption_index             18.7
kof_economic_globalisation       11.1
ccp_market_economy_provisions    10.9
ccp_civil_rights_provisions      10.9
ccp_information_access           10.9
ccp_equality_provisions          10.9
ccp_government_system             6.5
pts_statedept                     5.2
cow_code                          0.9
dtype: float64
  country_name  cow_code country_code  year  kof_economic_globalisation  \
0  Afghanistan     700.0          AFG  1990                   28.083757   
1  Afghanistan     700.0          AFG  1991                   27.990288   
2  Afghanistan     700.0          AFG  19

In [9]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "qog_clean.csv")
qog.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {qog.shape}")

# Get latest year from data
latest_year = str(int(qog['year'].max()))

# Update download log
update_entry(
    "GPI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: gpi_gpi."
)

update_entry(
    "PTS",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variables: gd_ptsa (Amnesty), gd_ptsh (HRW), gd_ptss (State Dept)."
)

update_entry(
    "OBS",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: ibp_obi (Open Budget Index)."
)

update_entry(
    "ND_GAIN",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variables: gain_gov (governance readiness), gain_read (readiness). Master PDF specifies these sub-scores not overall index."
)

update_entry(
    "BCI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: bci_bci."
)

update_entry(
    "HANSON_SIGMAN",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: lld_capacity. Double-counting caveat: incorporates V-Dem and other sources we use."
)

update_entry(
    "CCP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variables: ccp_syst, ccp_market, ccp_civil, ccp_infoacc, ccp_equal. Note: judicial independence and separation of powers sub-dimensions not clearly captured in QoG CCP variable subset — gap flagged."
)

update_entry(
    "PEI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: pei_peii_1. Per-election cadence — high missingness expected."
)

update_entry(
    "KOF_TRADE",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: dr_eg (Economic Globalisation). MISMATCH: master PDF calls for Trade Globalization subindex specifically. dr_eg covers trade+financial. Decision flagged in framework_decisions.md."
)

print("\nAll log entries updated.")
print_entry("KOF_TRADE")

Written: /Users/boulanger/Documents/governance-framework/data/processed/qog_clean.csv
Shape: (6889, 20)
[download_log] Updated entry for GPI
[download_log] Updated entry for PTS
[download_log] Updated entry for OBS
[download_log] Updated entry for ND_GAIN
[download_log] Updated entry for BCI
[download_log] Updated entry for HANSON_SIGMAN
[download_log] Updated entry for CCP
[download_log] Updated entry for PEI
[download_log] Updated entry for KOF_TRADE

All log entries updated.
  source_id: KOF_TRADE
  last_attempted_date: 2026-06-03
  last_successful_download_date: 2026-06-03
  data_as_of_date: 2025
  local_filename: qog_clean.csv
  latest_available_version: QoG Jan26
  no_update_reason: nan
  notes: Sourced via QoG Standard TS dataset. Variable: dr_eg (Economic Globalisation). MISMATCH: master PDF calls for Trade Globalization subindex specifically. dr_eg covers trade+financial. Decision flagged in framework_decisions.md.
